# M3: Adding Cross-Company Spillover Network Features

M3 extends M2 by adding **Diebold-Yilmaz dual-layer spillover network** features that capture cross-company dynamics the Magnificent 7.

**New features (from `04_build_spillover_network.py`):**
- `spillover_weighted_sent` — other companies' sentiment weighted by network influence on target
- `net_transmitter` — out-degree minus in-degree (leader vs follower in the network)
- `system_connectedness` — how tightly coupled the entire Mag7 system is
- `spillover_neg_asym` — bearish sentiment pressure from influential peers

**Carried from M2:**
- `surprise_pct`, `sent_mean`, `sent_trend`, `sent_delta`, `news_volume`

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

conn = sqlite3.connect('../data/hackathon.db')
print('DB connected')

## 1. Build Target + M1/M2 Features

Same construction as notebook 02 — build target variable and sentiment features from scratch.

In [ ]:
# ── Target: 5-day return direction after earnings ──
earnings = pd.read_sql('SELECT * FROM earnings', conn)
prices = pd.read_sql('SELECT * FROM daily_prices', conn)
prices['date'] = pd.to_datetime(prices['date'])
earnings['earnings_date'] = pd.to_datetime(earnings['earnings_date'])

records = []
for _, evt in earnings.iterrows():
    tk = evt['ticker']
    ed = evt['earnings_date']
    tk_prices = prices[prices['ticker'] == tk].sort_values('date')
    post = tk_prices[tk_prices['date'] > ed].head(5)
    if len(post) < 5:
        continue
    pre = tk_prices[tk_prices['date'] <= ed].tail(1)
    if pre.empty:
        continue
    p0 = pre['close'].values[0]
    p5 = post['close'].values[-1]
    ret_5d = (p5 - p0) / p0
    records.append({
        'ticker': tk, 'earnings_date': ed,
        'surprise_pct': evt['surprise_pct'],
        'ret_5d': ret_5d,
        'target': 1 if ret_5d > 0 else 0,
    })

df = pd.DataFrame(records).sort_values('earnings_date').reset_index(drop=True)
print(f'Total events: {len(df)}, Up: {df["target"].sum()}, Down: {(1-df["target"]).sum()}')

In [ ]:
# ── M2 Sentiment Features ──
articles = pd.read_sql('SELECT * FROM window_articles', conn)
articles['article_date'] = pd.to_datetime(articles['article_date'])
articles['earnings_date'] = pd.to_datetime(articles['earnings_date'])
daily_sent = pd.read_sql('SELECT * FROM daily_sentiment', conn)
daily_sent['date'] = pd.to_datetime(daily_sent['date'])

sent_features = []
for _, row in df.iterrows():
    tk, ed = row['ticker'], row['earnings_date']
    mask = (articles['ticker'] == tk) & (articles['earnings_date'] == ed)
    window = articles[mask].copy()
    if len(window) < 3:
        sent_features.append({'ticker': tk, 'earnings_date': ed,
                              'sent_mean': np.nan, 'sent_trend': np.nan,
                              'sent_delta': np.nan, 'news_volume': len(window)})
        continue
    sent_mean = window['polarity'].mean()
    daily_agg = window.groupby('article_date')['polarity'].mean().sort_index()
    n_days = len(daily_agg)
    if n_days >= 2:
        mid = n_days // 2
        sent_trend = daily_agg.iloc[mid:].mean() - daily_agg.iloc[:mid].mean()
    else:
        sent_trend = 0.0
    news_volume = len(window)
    quiet_start = ed - pd.Timedelta(days=37)
    quiet_end = ed - pd.Timedelta(days=30)
    quiet = daily_sent[(daily_sent['ticker'] == tk) &
                       (daily_sent['date'] >= quiet_start) &
                       (daily_sent['date'] <= quiet_end)]
    sent_delta = sent_mean - quiet['sentiment'].mean() if len(quiet) >= 3 else np.nan
    sent_features.append({'ticker': tk, 'earnings_date': ed,
                          'sent_mean': sent_mean, 'sent_trend': sent_trend,
                          'sent_delta': sent_delta, 'news_volume': news_volume})

df = df.merge(pd.DataFrame(sent_features), on=['ticker', 'earnings_date'])
print(f'M2 features computed: {len(df)} events')
print(f'NaN: {df[["sent_mean","sent_trend","sent_delta"]].isna().sum().to_dict()}')

## 2. Merge M3 Spillover Features

In [ ]:
# ── Load precomputed M3 features ──
m3_feat = pd.read_csv('../data/spillover/m3_features.csv')
m3_feat['earnings_date'] = pd.to_datetime(m3_feat['earnings_date'])
m3_cols = ['spillover_weighted_sent', 'net_transmitter', 'system_connectedness', 'spillover_neg_asym']

df = df.merge(m3_feat[['ticker', 'earnings_date'] + m3_cols],
              on=['ticker', 'earnings_date'], how='left')

print(f'After merge: {len(df)} events')
print(f'M3 feature coverage: {df[m3_cols[0]].notna().sum()} / {len(df)}')
print(f'\nM3 features NaN: {df[m3_cols].isna().sum().to_dict()}')
print(f'\nM3 feature stats:')
df[m3_cols].describe().round(4)

## 3. Walk-Forward Evaluation: M0 vs M1 vs M2 vs M3

All models evaluated on the same event set (events with complete M2 + M3 features) for fair comparison.

In [ ]:
# Drop events missing any feature
all_feat_cols = ['sent_mean', 'sent_trend', 'sent_delta', 'news_volume'] + m3_cols
df_eval = df.dropna(subset=all_feat_cols).reset_index(drop=True)
print(f'Events usable for M3: {len(df_eval)} (dropped {len(df) - len(df_eval)} with NaN)')

MIN_TRAIN = 40
y = df_eval['target'].values

# Feature sets
X_m1 = df_eval[['surprise_pct']].values
X_m2 = df_eval[['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume']].values
X_m3 = df_eval[['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume'] + m3_cols].values

# M0 predictions (rule-based)
m0_pred_all = (df_eval['surprise_pct'].values > 0).astype(int)

results = {name: {'true': [], 'pred': [], 'prob': []}
           for name in ['M0', 'M1', 'M2', 'M3']}

for t in range(MIN_TRAIN, len(df_eval)):
    y_test = y[t]
    
    # M0
    results['M0']['true'].append(y_test)
    results['M0']['pred'].append(m0_pred_all[t])
    results['M0']['prob'].append(float(m0_pred_all[t]))
    
    # M1
    m1 = LogisticRegression(random_state=42, max_iter=1000)
    m1.fit(X_m1[:t], y[:t])
    results['M1']['true'].append(y_test)
    results['M1']['pred'].append(m1.predict(X_m1[t:t+1])[0])
    results['M1']['prob'].append(m1.predict_proba(X_m1[t:t+1])[0, 1])
    
    # M2
    scaler2 = StandardScaler()
    X2_train = scaler2.fit_transform(X_m2[:t])
    X2_test = scaler2.transform(X_m2[t:t+1])
    m2 = LogisticRegression(random_state=42, max_iter=1000)
    m2.fit(X2_train, y[:t])
    results['M2']['true'].append(y_test)
    results['M2']['pred'].append(m2.predict(X2_test)[0])
    results['M2']['prob'].append(m2.predict_proba(X2_test)[0, 1])
    
    # M3
    scaler3 = StandardScaler()
    X3_train = scaler3.fit_transform(X_m3[:t])
    X3_test = scaler3.transform(X_m3[t:t+1])
    m3 = LogisticRegression(random_state=42, max_iter=1000)
    m3.fit(X3_train, y[:t])
    results['M3']['true'].append(y_test)
    results['M3']['pred'].append(m3.predict(X3_test)[0])
    results['M3']['prob'].append(m3.predict_proba(X3_test)[0, 1])

for m in results:
    for k in results[m]:
        results[m][k] = np.array(results[m][k])

print(f'Walk-forward test events: {len(results["M0"]["true"])}')

## 4. Metrics Comparison

In [ ]:
metrics = {}
for name in ['M0', 'M1', 'M2', 'M3']:
    yt = results[name]['true']
    yp = results[name]['pred']
    yprob = results[name]['prob']
    metrics[name] = {
        'Accuracy': accuracy_score(yt, yp),
        'Precision': precision_score(yt, yp, zero_division=0),
        'Recall': recall_score(yt, yp, zero_division=0),
        'F1': f1_score(yt, yp, zero_division=0),
        'AUC': roc_auc_score(yt, yprob),
    }

metrics_df = pd.DataFrame(metrics).T
print('Model Comparison (all on same test set):')
print(metrics_df.round(3).to_string())

# Trading returns
test_df = df_eval.iloc[MIN_TRAIN:].copy()
print(f'\nGross Trading Returns:')
for name in ['M0', 'M1', 'M2', 'M3']:
    strat_ret = np.where(results[name]['pred'] == 1,
                         test_df['ret_5d'].values,
                         -test_df['ret_5d'].values)
    gross = ((1 + strat_ret).cumprod()[-1] - 1) * 100
    print(f'  {name}: {gross:+.1f}%')
bh_gross = ((1 + test_df['ret_5d'].values).cumprod()[-1] - 1) * 100
print(f'  Buy & Hold: {bh_gross:+.1f}%')

In [ ]:
# Detailed classification reports
for name in ['M2', 'M3']:
    print(f'\n--- {name} Classification Report ---')
    print(classification_report(results[name]['true'], results[name]['pred'],
                                target_names=['Down', 'Up'], zero_division=0))

## 5. M3 Model Coefficients

In [ ]:
# Final model on all data for interpretation
scaler_final = StandardScaler()
X_m3_scaled = scaler_final.fit_transform(X_m3)
m3_final = LogisticRegression(random_state=42, max_iter=1000)
m3_final.fit(X_m3_scaled, y)

feat_names = ['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume'] + m3_cols
coef_df = pd.DataFrame({
    'Feature': feat_names,
    'Coefficient': m3_final.coef_[0],
    'Abs': np.abs(m3_final.coef_[0]),
}).sort_values('Abs', ascending=False)

print(f'Intercept: {m3_final.intercept_[0]:.4f}')
print(f'\nM3 Coefficients (standardized features):')
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))

# Also show M2 coefficients for comparison
scaler_m2f = StandardScaler()
X_m2_scaled = scaler_m2f.fit_transform(X_m2)
m2_final = LogisticRegression(random_state=42, max_iter=1000)
m2_final.fit(X_m2_scaled, y)
m2_feat_names = ['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume']
m2_coef_df = pd.DataFrame({
    'Feature': m2_feat_names,
    'M2_Coefficient': m2_final.coef_[0],
})
print(f'\nM2 Coefficients for comparison:')
print(m2_coef_df.to_string(index=False))

## 6. Visualizations: M0 vs M1 vs M2 vs M3

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('M0 vs M1 vs M2 vs M3: Progressive Model Comparison', fontsize=15, fontweight='bold')

colors_map = {'M0': '#F44336', 'M1': '#2196F3', 'M2': '#4CAF50', 'M3': '#FF9800'}

# ── 1. ROC Curves ──
ax = axes[0, 0]
for name in ['M1', 'M2', 'M3']:
    fpr, tpr, _ = roc_curve(results[name]['true'], results[name]['prob'])
    ax.plot(fpr, tpr, color=colors_map[name], linewidth=2,
            label=f'{name} (AUC={metrics[name]["AUC"]:.3f})')
# M0 as single point
cm0 = confusion_matrix(results['M0']['true'], results['M0']['pred'])
m0_fpr = cm0[0,1] / (cm0[0,0] + cm0[0,1]) if (cm0[0,0] + cm0[0,1]) > 0 else 0
m0_tpr = cm0[1,1] / (cm0[1,0] + cm0[1,1]) if (cm0[1,0] + cm0[1,1]) > 0 else 0
ax.plot(m0_fpr, m0_tpr, 'rs', markersize=12, label=f'M0 (AUC={metrics["M0"]["AUC"]:.3f})')
ax.plot([0,1],[0,1],'k--',alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves'); ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3)

# ── 2. Metrics Bar Chart ──
ax = axes[0, 1]
metric_names = ['Accuracy', 'F1', 'AUC']
x_pos = np.arange(len(metric_names))
width = 0.2
for i, name in enumerate(['M0', 'M1', 'M2', 'M3']):
    vals = [metrics[name][m] for m in metric_names]
    bars = ax.bar(x_pos + i*width, vals, width, label=name,
                  color=colors_map[name], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=7, fontweight='bold')
ax.set_xticks(x_pos + 1.5*width)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='Random')
ax.set_title('Classification Metrics'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── 3. M3 Feature Importance ──
ax = axes[1, 0]
coef_sorted = coef_df.sort_values('Coefficient')
colors_bar = ['#FF9800' if f in m3_cols else ('#4CAF50' if c > 0 else '#F44336')
              for f, c in zip(coef_sorted['Feature'], coef_sorted['Coefficient'])]
ax.barh(coef_sorted['Feature'], coef_sorted['Coefficient'], color=colors_bar, alpha=0.85)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('M3 Feature Coefficients (standardized)')
ax.set_xlabel('Coefficient (orange = new M3 features)')
ax.grid(alpha=0.3)

# ── 4. Cumulative Trading Returns ──
ax = axes[1, 1]
for name in ['M0', 'M1', 'M2', 'M3']:
    strat_ret = np.where(results[name]['pred'] == 1,
                         test_df['ret_5d'].values,
                         -test_df['ret_5d'].values)
    cum_ret = (1 + strat_ret).cumprod()
    gross = (cum_ret[-1] - 1) * 100
    lw = 2.5 if name == 'M3' else 1.5
    ax.plot(range(len(cum_ret)), cum_ret, color=colors_map[name],
            linewidth=lw, label=f'{name} ({gross:+.1f}%)')
# Buy & Hold
bh = (1 + test_df['ret_5d'].values).cumprod()
bh_gross = (bh[-1] - 1) * 100
ax.plot(range(len(bh)), bh, 'gray', linewidth=1, alpha=0.5,
        label=f'Buy & Hold ({bh_gross:+.1f}%)')
ax.axhline(y=1.0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Event #'); ax.set_ylabel('Cumulative Return')
ax.set_title('Simulated Trading Returns (Long/Short)')
ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/m0_m1_m2_m3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/m0_m1_m2_m3_comparison.png')

## 7. Where M3 Disagrees with M2 — Event-Level Analysis

In [ ]:
# Events where M3 and M2 make different predictions
analysis = test_df[['ticker', 'earnings_date', 'surprise_pct', 'ret_5d', 'target',
                     'sent_mean'] + m3_cols].copy()
analysis['m2_pred'] = results['M2']['pred']
analysis['m3_pred'] = results['M3']['pred']
analysis['m2_prob'] = results['M2']['prob']
analysis['m3_prob'] = results['M3']['prob']
analysis['m0_pred'] = results['M0']['pred']
analysis['actual'] = results['M3']['true']

disagree = analysis[analysis['m2_pred'] != analysis['m3_pred']].copy()
disagree['m3_correct'] = (disagree['m3_pred'] == disagree['actual']).astype(int)
disagree['m2_correct'] = (disagree['m2_pred'] == disagree['actual']).astype(int)

print(f'M3 vs M2 disagreements: {len(disagree)} out of {len(analysis)} events')
print(f'M3 correct when disagreeing: {disagree["m3_correct"].sum()} / {len(disagree)}')
print(f'M2 correct when disagreeing: {disagree["m2_correct"].sum()} / {len(disagree)}')

if len(disagree) > 0:
    print(f'\nDisagreement events:')
    show_cols = ['ticker', 'earnings_date', 'ret_5d', 'surprise_pct', 'sent_mean',
                 'system_connectedness', 'net_transmitter',
                 'm2_pred', 'm3_pred', 'actual', 'm3_correct']
    print(disagree[show_cols].to_string(index=False))

In [ ]:
# M3 PnL on disagreement events
if len(disagree) > 0:
    disagree['m3_pnl'] = np.where(disagree['m3_pred'] == 1,
                                   disagree['ret_5d'],
                                   -disagree['ret_5d']) * 100
    disagree['m2_pnl'] = np.where(disagree['m2_pred'] == 1,
                                   disagree['ret_5d'],
                                   -disagree['ret_5d']) * 100
    disagree['pnl_edge'] = disagree['m3_pnl'] - disagree['m2_pnl']
    
    print(f'\nM3 net PnL on disagreements: {disagree["m3_pnl"].sum():+.1f}%')
    print(f'M2 net PnL on disagreements: {disagree["m2_pnl"].sum():+.1f}%')
    print(f'M3 edge: {disagree["pnl_edge"].sum():+.1f}%')

## 8. System Connectedness Over Time

In [ ]:
# Plot system connectedness evolution
sc = df_eval[['earnings_date', 'system_connectedness']].drop_duplicates('earnings_date').sort_values('earnings_date')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sc['earnings_date'], sc['system_connectedness'], 'o-', color='#FF9800', linewidth=1.5, markersize=4)
ax.fill_between(sc['earnings_date'], sc['system_connectedness'], alpha=0.15, color='#FF9800')
ax.set_ylabel('System Connectedness')
ax.set_title('Mag7 System Connectedness Over Time (from DY Dual-Layer Network)')
ax.grid(alpha=0.3)

# Annotate notable periods
ax.axhline(y=sc['system_connectedness'].mean(), color='gray', linestyle='--', alpha=0.5, label=f'Mean={sc["system_connectedness"].mean():.2f}')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/system_connectedness_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/system_connectedness_timeline.png')

## 9. Summary Table

In [ ]:
# Final summary table
summary = metrics_df.copy()
for name in ['M0', 'M1', 'M2', 'M3']:
    strat_ret = np.where(results[name]['pred'] == 1,
                         test_df['ret_5d'].values,
                         -test_df['ret_5d'].values)
    gross = ((1 + strat_ret).cumprod()[-1] - 1) * 100
    summary.loc[name, 'Gross Return'] = f'{gross:+.1f}%'

summary.loc['Random'] = {'Accuracy': 0.500, 'Precision': '-', 'Recall': '-',
                          'F1': '-', 'AUC': 0.500, 'Gross Return': '-'}

print('\n' + '='*70)
print('FINAL MODEL COMPARISON')
print('='*70)
print(summary.to_string())
print(f'\nTest events: {len(results["M0"]["true"])}')

conn.close()
print('\nDone.')